# Run Video QA Streamlit App (v3) on Colab (GPU) via Cloudflare Tunnel

**Before running:** set the runtime to GPU --> *Runtime > Change runtime type > Hardware accelerator > GPU (T4)*.

v3 flow: **Analyze** shows only Normal/Anomalous + Summary; details (people/weapon/location/category/actions) are asked on demand in the VQA section.

Run the cells top-to-bottom. The last cell prints a public `https://*.trycloudflare.com` URL.

## 1. Clone the repository

In [ ]:
import os

REPO_URL = "https://github.com/sudeeprana8043-svg/Streamlit_project.git"
REPO_DIR = "/content/Streamlit_project"

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

%cd $REPO_DIR
!ls

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Configure model files

`MODEL_DIR` stays pointed at the repo's `model/` folder (ships the temporal-matched legacy files). This cell copies the `models3` artifacts in (skipping colliding encoders), pulls `temporal_adapter.pt` from the old `/models` folder, and loads the summarization checkpoint `checkpoint-414` into the `checkpoint-140` folder the app expects.

In [ ]:
import os, shutil

from google.colab import drive
drive.mount("/content/drive")

MODEL_LOCAL  = os.path.abspath("model_instruct")  # Changed to use Instruct-based models
os.makedirs(MODEL_LOCAL, exist_ok=True)

# First, copy from local model/ folder (these files exist in git)
MODEL_REPO = os.path.abspath("model")
if os.path.exists(MODEL_REPO):
    print(f"Copying from local model/ folder to model_instruct/...")
    for fname in os.listdir(MODEL_REPO):
        src = os.path.join(MODEL_REPO, fname)
        dst = os.path.join(MODEL_LOCAL, fname)
        if os.path.isfile(src) and not os.path.exists(dst):
            shutil.copy(src, dst)
            print(f"  Copied {fname}")

# Then copy from Drive models3 (for newer models)
DRIVE_MODELS = "/content/drive/MyDrive/models3"
if os.path.exists(DRIVE_MODELS):
    print(f"\nCopying from Drive models3 to model_instruct/...")
    for fname in sorted(os.listdir(DRIVE_MODELS)):
        src = os.path.join(DRIVE_MODELS, fname)
        if not os.path.isfile(src):
            continue
        dst = os.path.join(MODEL_LOCAL, fname)
        if not os.path.exists(dst):
            shutil.copy(src, dst)
            print(f"  Copied {fname}")

# Copy legacy files from old Drive models folder
OLD_DRIVE_MODELS = "/content/drive/MyDrive/models"
if os.path.exists(OLD_DRIVE_MODELS):
    print(f"\nCopying legacy files from old Drive models folder...")
    for legacy in ["temporal_adapter.pt", "binary_model.pkl", "model_config.pkl"]:
        dst = os.path.join(MODEL_LOCAL, legacy)
        src = os.path.join(OLD_DRIVE_MODELS, legacy)
        if not os.path.exists(dst) and os.path.exists(src):
            shutil.copy(src, dst)
            print(f"  Copied {legacy}")

# Copy summary checkpoint
SUMMARY_CKPT_DRIVE = "/content/drive/MyDrive/Project_VLM/ucf_qwen_v9_qformer/checkpoint-414"
SUMMARY_CKPT_LOCAL = os.path.join(MODEL_LOCAL, "checkpoint-140")
if os.path.isdir(SUMMARY_CKPT_DRIVE):
    shutil.copytree(SUMMARY_CKPT_DRIVE, SUMMARY_CKPT_LOCAL, dirs_exist_ok=True)
    os.environ["SUMMARY_CHECKPOINT"] = SUMMARY_CKPT_LOCAL
    os.environ["LORA_CHECKPOINT"] = SUMMARY_CKPT_LOCAL
    print(f"\nLoaded checkpoint-414 weights into {SUMMARY_CKPT_LOCAL}")
else:
    print(f"\nWarning: Summary checkpoint not found at {SUMMARY_CKPT_DRIVE}")

os.environ["MODEL_DIR"] = MODEL_LOCAL
os.environ["BINARY_MODEL_DIR"] = MODEL_LOCAL

print(f"\nChecking required files in {MODEL_LOCAL}:")
for f in ["le_weapon.pkl", "le_location.pkl", "le_people.pkl", "le_super.pkl", 
          "multiclass_bundle.pkl", "simple_adapter.pt", "lora_model.pt", "input_dim.pkl"]:
    status = "ok" if os.path.exists(os.path.join(MODEL_LOCAL, f)) else "MISSING"
    print(f"  {f}: {status}")

## 4. Download the Cloudflare tunnel binary

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared --version

## 5. Launch Streamlit (v3) + public tunnel

In [ ]:
import subprocess, time, os, sys

PORT = 8501
APP = "streamlit_app_v3.py"  # v3 = analyze (status+summary) then on-demand VQA

streamlit_proc = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", APP,
        "--server.port", str(PORT),
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
    ],
    stdout=open("/content/streamlit.log", "w"),
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

print(f"Starting {APP}... (give it ~20s to boot and load models)")
time.sleep(20)
print("Recent Streamlit log:")
!tail -n 20 /content/streamlit.log

print("\n=== Public URL will appear below (look for *.trycloudflare.com) ===\n")
!./cloudflared tunnel --url http://localhost:$PORT